# 🤖📰 Treinar a IA de Notícias — SmartTrader

Treina/usa a **IA que lê notícias e decide direção** (comprar/vender) — o coração da sua ideia.
Rode no **Google Colab** com **GPU** (*Ambiente de execução → Alterar tipo → GPU*).

### ⚠️ Honestidade primeiro
- **FinBERT** mede o **sentimento** de texto financeiro (já vem treinado).
- 'Treinar' = **especializar (fine-tune)** o FinBERT num dataset financeiro real, na GPU.
- Sentimento **não é direção**. Quem vira 'petróleo↑' em 'USDCAD↓' é a camada macro (Passo 4).
- Não garante lucro — é filtro de qualidade. Valide em demo antes de real.

### 🛠️ Se der erro de `torch`
Não reinstale o torch! O Colab já tem um compatível. Se aparecer erro de torch/torchvision:
**Ambiente de execução → Reiniciar sessão** e rode de novo a partir do Passo 1.


## Passo 1 — Instalar dependências
Instala SÓ o que falta (transformers + datasets). **Não** reinstala o torch — isso evita o erro de versão.


In [ ]:
# NÃO instale 'torch' aqui: o Colab já vem com torch+torchvision compatíveis.
!pip install -q -U transformers datasets

import torch
print('Torch:', torch.__version__, '| GPU:', torch.cuda.is_available())


## Passo 2 — Usar o FinBERT pronto (já funciona)
Carrega o modelo **direto** (sem `pipeline`, que puxa o torchvision e dá conflito).
Mede o sentimento de manchetes.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()

def sentimento(texto):
    inp = tok(texto, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inp).logits
    probs = torch.softmax(logits, dim=1)[0]
    i = int(probs.argmax())
    return model.config.id2label[i], float(probs[i])

manchetes = [
    'Oil prices spike after OPEC supply cut amid Middle East war',
    'Tech stocks rally as inflation cools and growth beats forecasts',
    'Markets plunge as recession fears and conflict escalate',
]
for m in manchetes:
    lab, sc = sentimento(m)
    print(f'{lab:9s} ({sc:.2f})  <-  {m}')


## Passo 3 — (GPU) Fine-tunar o FinBERT no Financial PhraseBank
Especializa o modelo num dataset rotulado por especialistas. Alguns minutos na GPU.
Opcional — pule se só quiser usar o Passo 2.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

# Dataset financeiro real (0=negativo, 1=neutro, 2=positivo)
ds = load_dataset('financial_phrasebank', 'sentences_50agree', trust_remote_code=True)['train']
ds = ds.train_test_split(test_size=0.2, seed=42)

MODEL = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(MODEL)
def prep(b): return tok(b['sentence'], truncation=True, padding='max_length', max_length=128)
ds = ds.map(prep, batched=True).rename_column('label', 'labels')
ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': float((preds == p.label_ids).mean())}

args = TrainingArguments(output_dir='/content/finbert_ft', num_train_epochs=1,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy='epoch', logging_steps=50, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=ds['train'],
    eval_dataset=ds['test'], compute_metrics=metrics)
trainer.train()
print('Avaliação:', trainer.evaluate())
trainer.save_model('/content/finbert_ft'); tok.save_pretrained('/content/finbert_ft')
print('Modelo salvo em /content/finbert_ft')


## Passo 4 — De SENTIMENTO para DIREÇÃO (a sua ideia)
A notícia vira **comprar/vender** por ativo, tratando **forças conflitantes**
(crise no petróleo fortalece o CAD, mas o safe-haven fortalece o USD). Versão resumida
e auditável do `news_mapper.py` do projeto.


In [ ]:
THEME_KW = {
  'oil': ['oil','crude','opec','brent','wti','petroleum','petroleo'],
  'risk_off': ['war','crisis','conflict','recession','crash','plunge'],
  'risk_on': ['rally','optimism','growth beats','soar'],
}
EXPO = {
  'USDCAD': {'oil': -0.7, 'risk_off': +0.6},
  'XAUUSD': {'risk_off': +0.8, 'oil': +0.2},
  'USOIL':  {'oil': +1.0},
  'SPX':    {'risk_off': -0.8, 'risk_on': +0.7},
}

def detectar_temas(textos):
    t = ' '.join(textos).lower(); temas = {}
    for tema, kws in THEME_KW.items():
        hits = sum(t.count(k) for k in kws)
        if hits: temas[tema] = min(1.0, hits/(hits+1))
    return temas

def direcao(simbolo, temas):
    net = sum(EXPO.get(simbolo,{}).get(tm,0.0)*f for tm,f in temas.items())
    bias = 1 if net > 0.15 else (-1 if net < -0.15 else 0)
    return bias, min(1.0, abs(net)), net

noticia = ['Oil prices spike after OPEC supply cut amid Middle East war']
lab, sc = sentimento(noticia[0])    # usa o FinBERT do Passo 2
print('Sentimento FinBERT:', lab, f'({sc:.2f})')
temas = detectar_temas(noticia); print('Temas:', temas)
for s in ['USOIL','XAUUSD','USDCAD','SPX']:
    b, c, net = direcao(s, temas)
    print(f"  {s}: {({1:'COMPRA',-1:'VENDA',0:'NEUTRO'}[b])}  (conf {c:.2f}, net {net:+.2f})")


## Passo 5 — Plugar no bot (depois)
1. Baixe a pasta `/content/finbert_ft` (modelo fine-tunado).
2. No projeto, em `NewsBiasEngine._score_sentiment` carregue esse modelo (igual ao Passo 2)
   e em `_interpret_macro` use a lógica do Passo 4 (ou o `news_mapper.py`).
3. Ligue `USE_AI=true` no `.env` e rode na conta **DEMO** primeiro.

> 🎯 FinBERT dá **sentimento**; a camada macro dá **direção**. Juntos viram o viés que a
> técnica confirma (Modo A). Sentimento de notícia **não é bola de cristal** — valide em demo.
